In [8]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
from statsmodels.stats.multitest import multipletests
import time

EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
EA_META = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR = r"C:\Users\user\Downloads\GSE148812_clean"

print("Imports done.")

Imports done.


In [9]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)

encoded_df = pd.read_csv(EA_GENO)
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_ids = encoded_df.columns[1:].tolist()

keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])
print("Autosomal probes:", keep_mask.sum())

X_snp = encoded_df[encoded_df.columns[1:]].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp.T
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp
gc.collect()

print("X_auto shape:", X_auto.shape)

Autosomal probes: 233721
X_auto shape: (1595, 233721)


In [10]:
meta_df = pd.read_csv(EA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()

print("Metadata shape:", meta_df.shape)
print("Smoking status:", meta_df["smoking_status"].value_counts())
print("Missing:", meta_df[["age", "gender", "smoking_status"]].isna().sum())

# outcome
Y = (meta_df["smoking_status"] == "Smoker").astype(np.float64).values
print("Y distribution:", np.unique(Y, return_counts=True))

# confounders
age_std = (meta_df["age"].astype(float) - meta_df["age"].astype(float).mean()) / meta_df["age"].astype(float).std()
gender_binary = (meta_df["gender"] == "Male").astype(np.float64).values
print("Gender distribution:", np.unique(gender_binary, return_counts=True))

Metadata shape: (1595, 9)
Smoking status: smoking_status
Smoker        865
Non-smoker    730
Name: count, dtype: int64
Missing: age               0
gender            0
smoking_status    0
dtype: int64
Y distribution: (array([0., 1.]), array([730, 865]))
Gender distribution: (array([0., 1.]), array([937, 658]))


In [11]:
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Monomorphic excluded:", (~valid_mask).sum())
print("Informative retained:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]

del X_float
gc.collect()

print("X_std shape:", X_std.shape)

pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)
print("Explained variance ratio:", pca.explained_variance_ratio_)

X_conf = np.hstack([
    pcs,
    age_std.values.reshape(-1, 1) if hasattr(age_std, 'values') else age_std.reshape(-1, 1),
    gender_binary.reshape(-1, 1)
]).astype(np.float64)

print("X_conf shape:", X_conf.shape)  # expect (1595, 12)

Monomorphic excluded: 106030
Informative retained: 127691
X_std shape: (1595, 127691)
Explained variance ratio: [0.02228285 0.00572736 0.0030517  0.00301088 0.0029775  0.0029516
 0.00292109 0.00291098 0.00289396 0.00286554]
X_conf shape: (1595, 12)


In [12]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

print("Running single test iteration...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_std, Y, X_conf, random_state=0)
print(f"Completed in {time.time()-start:.1f}s")

for thresh in [0.01, 0.001, 0.0001]:
    print(f"Raw p < {thresh}: {(pvals_test < thresh).sum()} SNPs")

Running single test iteration...
Completed in 28.4s
Raw p < 0.01: 3771 SNPs
Raw p < 0.001: 833 SNPs
Raw p < 0.0001: 333 SNPs


In [13]:
n_repeats = 30
threshold = 0.001
n_snps = X_std.shape[1]

significant_counts = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        print(f"  Repeat {rep+1}/{n_repeats} — {(time.time()-start)/60:.1f} min")

stability_fraction = significant_counts / n_repeats

stability_df_ea = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df_ea.to_csv(os.path.join(OUT_DIR, "v2_stability_results.csv"), index=False)

print(f"\nTotal: {(time.time()-start)/60:.1f} min")
for thresh in [0.5, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {thresh:.0%} stability: {(stability_fraction >= thresh).sum()} SNPs")

Running 30 stability repeats (p<0.001)...
  Repeat 5/30 — 2.1 min
  Repeat 10/30 — 4.2 min
  Repeat 15/30 — 6.2 min
  Repeat 20/30 — 8.3 min
  Repeat 25/30 — 10.4 min
  Repeat 30/30 — 12.5 min

Total: 12.5 min
  >= 50% stability: 716 SNPs
  >= 70% stability: 472 SNPs
  >= 80% stability: 381 SNPs
  >= 90% stability: 311 SNPs
  >= 100% stability: 57 SNPs


In [7]:
shortlist_ea = stability_df_ea[stability_df_ea["stability_fraction"] >= 0.90].copy()
shortlist_ea["core_name"] = shortlist_ea["probe_id"].map(strip_suffix)

pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_ea = shortlist_ea.merge(pos_lookup, on="core_name", how="left")
shortlist_ea = shortlist_ea[~shortlist_ea["Chr"].isin(non_autosomal)].copy()

print("90% stable SNPs:", len(shortlist_ea))
print(shortlist_ea[["probe_id", "Chr", "MapInfo", "stability_fraction"]].sort_values(["Chr", "MapInfo"]))

# load genotype vectors for LD pruning
encoded_df_ea = pd.read_csv(EA_GENO)
probe_rows_ea = encoded_df_ea[encoded_df_ea["probe_id"].isin(set(shortlist_ea["probe_id"]))].copy()
probe_rows_ea = probe_rows_ea.set_index("probe_id").reindex(shortlist_ea["probe_id"].tolist())
X_shortlist = probe_rows_ea.to_numpy(dtype=np.float64).T
probe_id_to_idx_ea = {pid: i for i, pid in enumerate(shortlist_ea["probe_id"].tolist())}

del encoded_df_ea, probe_rows_ea
gc.collect()

def get_geno_ea(pid):
    return X_shortlist[:, probe_id_to_idx_ea[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno_ea(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno_ea(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_ea = greedy_ld_prune(shortlist_ea, r2_thresh=0.2)
shortlist_pruned_ea = shortlist_ea[shortlist_ea["probe_id"].isin(retained_ea)].copy()
shortlist_pruned_ea = shortlist_pruned_ea.sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"\nAfter LD pruning: {len(shortlist_pruned_ea)} SNPs")
shortlist_pruned_ea.to_csv(os.path.join(OUT_DIR, "v2_shortlist_ld_pruned.csv"), index=False)
print(shortlist_pruned_ea[["probe_id", "Chr", "MapInfo"]].to_string())

90% stable SNPs: 20
                            probe_id Chr      MapInfo  stability_fraction
7         exm144193-0_B_R_2060145058   1  207195568.0            0.966667
5         exm935491-0_T_R_1918372056  11   68562288.0            1.000000
19       exm1006784-0_B_R_1922546096  12   53207439.0            0.900000
8        exm1071911-0_B_F_2060125197  13   73335638.0            0.966667
1        exm1094587-0_B_R_1922736292  14   25043951.0            1.000000
3        exm1094597-0_T_R_2060139141  14   25045386.0            1.000000
12       exm1215705-0_T_F_1921777411  16    5097934.0            0.933333
0   exm-rs4257308-131_B_F_1990483144  18   58109190.0            1.000000
11        exm244139-0_T_R_1918920506   2  178098917.0            0.933333
9        exm2273028-0_B_F_1984844263  22   31333631.0            0.966667
6        exm2264739-0_T_F_2060130663  22   50482126.0            1.000000
13        exm289607-0_T_F_2060424793   3   10146173.0            0.933333
10        exm32611

In [14]:
import json
import numpy as np
import pandas as pd
import os

EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
EA_META = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint2_metadata_sample_filtered.csv"
EA_OUT = r"C:\Users\user\Downloads\GSE148812_clean"

ea_shortlist = pd.read_csv(os.path.join(EA_OUT, "v2_shortlist_ld_pruned_v2.csv"))
pruned_ids_ea = ea_shortlist["probe_id"].tolist()
col_names_ea = pruned_ids_ea + ["smoking_status"]

encoded_df = pd.read_csv(EA_GENO)
sample_ids = encoded_df.columns[1:].tolist()

meta_df = pd.read_csv(EA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()
Y = (meta_df["smoking_status"] == "Smoker").astype(np.float64).values

probe_rows = encoded_df[encoded_df["probe_id"].isin(set(pruned_ids_ea))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(pruned_ids_ea)
X_pc_ea = probe_rows.to_numpy(dtype=np.float64).T

X_pc_full_ea = np.hstack([X_pc_ea, Y.reshape(-1, 1)])
print("PC input shape:", X_pc_full_ea.shape)  # expect (1595, 53)

np.save(os.path.join(EA_OUT, "v2_pc_input_v2.npy"), X_pc_full_ea)
with open(os.path.join(EA_OUT, "v2_pc_col_names_v2.json"), "w") as f:
    json.dump(col_names_ea, f)
print("Saved.")

PC input shape: (1595, 53)
Saved.
